In [3]:
import numpy as np
import pandas as pd

# Load the cleaned dataset after EDA_3
df_final = pd.read_csv(r'C:\dev\clinicaltrials-study\data\processed\df_EDA_3.csv', keep_default_na=False)

print(df_final.shape)

# Inspect columns
df_final.columns

(278762, 96)


Index(['nct_id', 'enrollment', 'overall_status', 'number_of_arms', 'has_dmc',
       'has_expanded_access', 'is_fda_regulated_drug',
       'is_fda_regulated_device', 'duration_of_study', 'phase_1', 'phase_2',
       'phase_3', 'phase_4', 'phase_unknown', 'intervention_behavioral',
       'intervention_biological', 'intervention_combination_product',
       'intervention_device', 'intervention_diagnostic_test',
       'intervention_dietary_supplement', 'intervention_drug',
       'intervention_genetic', 'intervention_other', 'intervention_procedure',
       'intervention_radiation', 'intervention_count',
       'has_multiple_intervention_types', 'condt_cancers',
       'condt_cardiovascular_diseases', 'condt_dental_disorders',
       'condt_dermatological_disorders', 'condt_endocrine/metabolic_disorders',
       'condt_gastrointestinal_disorders', 'condt_genetic_disorders',
       'condt_infectious_diseases', 'condt_mental_disorders',
       'condt_musculoskeletal_disorders', 'condt_ne

In [4]:
# Drop columns used only during EDA (not needed for modeling)
eda_only_cols = [
    'role_collaborator', 'role_lead',      # sponsor role (dropped after EDA)
    'duration_bins', 'arms_capped',        # engineered bins for EDA
    'duration_of_study', 'enrollment'      # raw duration/enrollment (we keep log values instead)
]

df_final.drop(columns=eda_only_cols, inplace=True, errors="ignore")

In [ ]:
# Create One-Hot Encoded Dataset for LogReg
# 로지스틱 회귀 모델링 위한 최종데이터셋 전처리 (새로운 데이터프레임)
# Drop grouped categorical columns, keep one-hot encoded flags
# 이미 원핫인코딩 통해 수치화된 더미변수들이 존재하는데, 원본 범주형 컬럼(_grouped)이 남아있으면 정보가 중복됨
# 다중 공선성 방지를 위해 제거함
df_final_onehot = df_final.drop(columns=[c for c in df_final if c.endswith('grouped')])
print("One-Hot Dataset Shape:", df_final_onehot.shape)

One-Hot Dataset Shape: (278762, 80)


In [6]:
# Create Grouped Dataset for xgBoost
# xgboost 모델링을 위한 최종데이터셋 전처리 (새로운 데이터프레임)
# Define the core columns we want to retain in grouped dataset
# 그룹화된 데이터 세트에서 유지할 핵심 변수 정의 = EDA통해 검증된 핵심 변수
columns_to_keep = [
    'nct_id', 'overall_status',
    'number_of_arms', 'intervention_count',
    'has_multiple_intervention_types',
    'log_enrollment', 'log_duration',
    'high_enroll_flag_975', 'high_enroll_flag_99',
    'has_dmc', 'has_expanded_access', 'healthy_volunteers',
    'is_fda_regulated_drug', 'is_fda_regulated_device'
]

# Keep grouped categorical variables + selected numeric/flags
# 그룹화된 범주형 변수와 선택된 숫자/플래그를 유지
# _grouped 컬럼들을 원본 형태로 유지한것은, 트리 모델이 "암 이면서 임상2상인 경우"와 같은 복합 조건들을 훨씬 더 효율적으로 찾게하기 위함
df_final_grouped = df_final[columns_to_keep + [c for c in df_final.columns if c.endswith('grouped')]]
print("Grouped Dataset Shape:", df_final_grouped.shape)

Grouped Dataset Shape: (278762, 24)


In [7]:
# Quick sanity check for null values
print("Nulls in One-Hot:", df_final_onehot.isnull().sum().sum())
print("Nulls in Grouped:", df_final_grouped.isnull().sum().sum())

Nulls in One-Hot: 0
Nulls in Grouped: 0


In [11]:
# Save final datasets
df_final_onehot.to_csv('../../data/final/df_final_onehot.csv', index=False)
df_final_grouped.to_csv('../../data/final/df_final_grouped.csv', index=False)

print("✅ Final datasets saved:")
print("One-Hot Encoded:", df_final_onehot.shape, "→ ../data/final/df_final_onehot.csv")
print("Grouped:", df_final_grouped.shape, "→ ../data/final/df_final_grouped.csv")

✅ Final datasets saved:
One-Hot Encoded: (278762, 80) → ../data/final/df_final_onehot.csv
Grouped: (278762, 24) → ../data/final/df_final_grouped.csv
